In [ ]:
# uv add kagglehub[pandas-datasets]

In [5]:
from pathlib import Path
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import shutil

In [3]:
# Download latest version
load_path = kagglehub.dataset_download("boltzmannbrain/nab")

print("Path to dataset files:", load_path)

Path to dataset files: /Users/mayazwang/.cache/kagglehub/datasets/boltzmannbrain/nab/versions/1


In [6]:
# Target folder for the dataset.
# Note: Jupyter's cwd is the notebook's folder (already "Anomaly_detection"),
# so handle both cases to avoid creating Anomaly_detection/Anomaly_detection/data.
cwd = Path.cwd()
file_path = cwd / "data" if cwd.name == "Anomaly_detection" else cwd / "Anomaly_detection/data/"
file_path.mkdir(parents=True, exist_ok=True)

# Copy the downloaded NAB files from the kagglehub cache into the data folder
shutil.copytree(load_path, file_path / "nab", dirs_exist_ok=True,
                ignore=shutil.ignore_patterns("__MACOSX"))
print("Dataset copied to:", file_path / "nab")

# NAB is a folder of many CSVs (one time series each), so load them individually
datasets = {
    f"{csv.parent.name}/{csv.stem}": pd.read_csv(csv, parse_dates=["timestamp"])
    for csv in sorted((file_path / "nab").rglob("*.csv"))
}
print(f"Loaded {len(datasets)} time series")
for name in list(datasets)[:5]:
    print(name, datasets[name].shape)

Dataset copied to: /Users/mayazwang/Files/00_Learning/_AI_security/Anomaly_detection/data/nab
Loaded 58 time series
artificialNoAnomaly/art_daily_no_noise (4032, 2)
artificialNoAnomaly/art_daily_perfect_square_wave (4032, 2)
artificialNoAnomaly/art_daily_small_noise (4032, 2)
artificialNoAnomaly/art_flatline (4032, 2)
artificialNoAnomaly/art_noisy (4032, 2)


In [7]:
# Example: pick one series
df = datasets["realKnownCause/machine_temperature_system_failure"]
df.head()

,timestamp,value
0,2013-12-02 21:15:00,73.967322
1,2013-12-02 21:20:00,74.935882
2,2013-12-02 21:25:00,76.124162
3,2013-12-02 21:30:00,78.140707
4,2013-12-02 21:35:00,79.329836


In [ ]:
# Download more datasets and copy them into data/<name>/
extra_datasets = {
    "creditcardfraud": "mlg-ulb/creditcardfraud",
    "network_traffic": "ziya07/network-traffic-anomaly-detection-dataset",
}

for name, handle in extra_datasets.items():
    src = kagglehub.dataset_download(handle)
    dst = file_path / name
    shutil.copytree(src, dst, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__MACOSX"))
    print(f"{handle} -> {dst}")
    for f in sorted(dst.rglob("*")):
        if f.is_file():
            print("   ", f.relative_to(dst), f"{f.stat().st_size / 1e6:.1f} MB")